# Woodward-Colella Double Blast Wave (1D)

This notebook runs the Woodward-Colella interacting blast-wave benchmark: two strong shocks, launched from thin high-pressure regions at either end of the tube, propagate inward and collide near `x = 0.7`. The collision produces a sharp, briefly very fast-moving contact/shock complex, which is why this case needs the adaptive `timestep` hook (`woodwardColellaCase.timestep`) and a tighter CFL factor than the other 1D examples -- a `dt` picked from the quiescent initial state would be far too large once the blasts meet.

Like `sod_1d.ipynb`, this notebook calls the real case code (`warpSPH.cases.woodwardColella.woodwardColellaCase`) rather than re-deriving it, and keeps the step loop unrolled in a cell instead of hiding it inside `warpSPH.runner.run()`. Because `timestep` re-picks `dt` every step, the loop below is a `while t < tLimit` rather than a fixed `range(nSteps)`, exactly like `03-kidder-isentropic-compression.ipynb`'s (this case has no `postStep`, so it is the same loop minus that one call). Plotting calls `drawWoodwardColella` (the same per-frame redraw `woodwardColellaCase.setupPlot`/`updatePlot` use internally, exported directly from `warpSPH.cases.woodwardColella` for this) rather than going through the `Case` hooks' `openWindow`/`pumpEvents`, which does not live-update reliably inside a Jupyter cell in this environment.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/05-Wodward_Colella_Double_Blastwave.gif)


In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.woodwardColella import woodwardColellaCase, WOODWARD_REGIONS, drawWoodwardColella
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `05-woodward-colella.py`, made explicit and editable here.
# `woodwardColellaCase.defaults`/`.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=woodwardColellaCase.name, scheme=woodwardColellaCase.scheme,
                params=dict(woodwardColellaCase.params)) \
    .merged(**woodwardColellaCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=2000,
    dim=1,
    L=2.0,

    # --- time stepping ---------------------------------------------------
    tLimit=0.038,
    # No `dt` here -- `woodwardColellaCase` has a `timestep` hook, so `dt`
    # starts from the CFL-derived value the sampler leaves in `config.dt` and
    # is re-picked every step below.
    cflFactor=0.2,

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=50,
    store=False,

    # --- Woodward-Colella's own knobs -----------------------------------------
    params=dict(
        gamma=1.4,
        regions=WOODWARD_REGIONS,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`woodwardColellaCase.buildSystem` -> `sampleShockRegions1D`), not
# re-derived here.
ctx = buildContext(woodwardColellaCase, spec)
woodwardColellaCase.configureScheme(ctx)
system = woodwardColellaCase.buildSystem(ctx)
runningState = system.initializeNewState()

for region in ctx.param('regions'):
    print(region)


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct drawWoodwardColella + plt.subplots(), not woodwardColellaCase.setupPlot
# -- see the intro cell for why.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    fig, axis = plt.subplots(2, 2, figsize=(9, 6), squeeze=False)
    drawWoodwardColella(ctx, runningState, (fig, axis))
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = woodwardColellaCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=woodwardColellaCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
# `woodwardColellaCase.timestep` is what makes this a `while t < tLimit` loop
# rather than a fixed `range(nSteps)` -- see the intro cell.
dt0 = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
storeSteps = max(1, int(spec.exportInterval / dt0)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
tq = tqdm(total=1000, leave=True)
i = 0
t = 0.0
while t < spec.tLimit:
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    ctx.config.dt = woodwardColellaCase.timestep(ctx, runningState)
    # -------------------------------------------------------------------------

    t = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = woodwardColellaCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=t))
    tq.n = min(1000, int(t / spec.tLimit * 1000))
    tq.set_description(f"t: {t:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))
    tq.refresh()

    final = t >= spec.tLimit
    if fig is not None and (i % spec.plotInterval == 0 or final):
        drawWoodwardColella(ctx, runningState, (fig, axis))
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or final):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=woodwardColellaCase.extraFields)

    i += 1
tq.close()


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
